# Stage 3.1: Model Development & Comparisons

This notebook covers the baseline model training and comparison phase of our project. 
Our objectives are to:
1. Train five classification models (Logistic Regression, Decision Tree, Random Forest, XGBoost, and LightGBM) on SMOTE and SMOTEENN datasets.
2. Evaluate model performance on the original imbalanced test set.
3. Focus on metrics like **Recall (class 1)** and **ROC-AUC** to balance false negatives against general discriminative power.
4. Compare performance metrics and plot multi-model ROC curves to select the winning architecture.

In [ ]:
import os
import sys
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Add src folder to system path to import modules
sys.path.append('../src')
from evaluate import calculate_metrics, plot_roc_curves
from train import load_processed_data, get_base_models

sns.set_theme(style="whitegrid")

## 1. Load Processed Datasets

In [ ]:
# Load data using helper function from training module
X_train_smote, y_train_smote, X_train_smoteenn, y_train_smoteenn, X_test, y_test = load_processed_data('../data/processed')

print("SMOTE Train set shape:", X_train_smote.shape)
print("SMOTEENN Train set shape:", X_train_smoteenn.shape)
print("Test set shape:", X_test.shape)

## 2. Train Base Models on SMOTE-Balanced Set

In [ ]:
results = []
models_smote = get_base_models()
trained_models_smote = {}

for name, model in models_smote.items():
    print(f"Training {name} on SMOTE...")
    model.fit(X_train_smote, y_train_smote)
    trained_models_smote[name] = model
    
    # Predict
    y_pred = model.predict(X_test)
    y_prob = model.predict_proba(X_test)[:, 1] if hasattr(model, 'predict_proba') else None
    
    # Calculate Metrics
    metrics = calculate_metrics(y_test, y_pred, y_prob)
    metrics['Model'] = name
    metrics['Balancing'] = 'SMOTE'
    results.append(metrics)

## 3. Train Base Models on SMOTEENN-Balanced Set

In [ ]:
models_smoteenn = get_base_models()
trained_models_smoteenn = {}

for name, model in models_smoteenn.items():
    print(f"Training {name} on SMOTEENN...")
    model.fit(X_train_smoteenn, y_train_smoteenn)
    trained_models_smoteenn[name] = model
    
    # Predict
    y_pred = model.predict(X_test)
    y_prob = model.predict_proba(X_test)[:, 1] if hasattr(model, 'predict_proba') else None
    
    # Calculate Metrics
    metrics = calculate_metrics(y_test, y_pred, y_prob)
    metrics['Model'] = name
    metrics['Balancing'] = 'SMOTEENN'
    results.append(metrics)

## 4. Performance Comparisons
We evaluate models using a metric score combining `Recall + ROC-AUC` to ensure the final model maintains solid discriminative boundaries while catching as many churn cases as possible.

In [ ]:
results_df = pd.DataFrame(results)
results_df['Selection_Score'] = results_df['Recall'] + results_df['ROC-AUC']

# Display comparisons
results_df_sorted = results_df.sort_values(by='Selection_Score', ascending=False)
results_df_sorted

## 5. ROC Curves Comparison

In [ ]:
# Plot ROC for SMOTE trained models
print("ROC Curves - SMOTE")
plot_roc_curves(trained_models_smote, X_test, y_test)

# Plot ROC for SMOTEENN trained models
print("ROC Curves - SMOTEENN")
plot_roc_curves(trained_models_smoteenn, X_test, y_test)

## 6. Selection Summary
Based on the results, we extract the best architecture and balancing combination. We will tune this combination using Optuna in the next notebook.

In [ ]:
winner = results_df_sorted.iloc[0]
print(f"Best Base Model: {winner['Model']}")
print(f"Best Balancing: {winner['Balancing']}")
print(f"Recall score: {winner['Recall']:.4f}")
print(f"ROC-AUC score: {winner['ROC-AUC']:.4f}")